# Part c: Activation functions, depth and width

We compare the sigmoid, the ReLU and the leaky ReLU in the hidden layers, for networks of different depth and width.
The data and scaling are the same as in part b. Unlike part b, we do not use a validation set.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # add Code/ so `src` can be imported

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import runge, make_runge_split, scale_data
from src.functions import *
from src.plotting import line_plot, heatmap, save_fig
from src.network import NeuralNetwork
from src.optimizers import Adam

## Data 

In [ ]:
# Recreating the data set from project 1
X_train, X_test, y_train, y_test = make_runge_split(n=100, sigma=0.1, test_size=0.2, seed=2026)
#Scaling and centering the data
X_train_s, X_test_s, x_mean, x_std = scale_data(X_train, X_test)

# 300 evenly spaced points between -1 and 1, for smooth curces in the plots
x_plot = np.linspace(-1, 1, 300).reshape(-1, 1)
# the true Runge function 
y_true = runge(x_plot)

## Activations and architectures


In [ ]:
# Activation functions and their derivatives
activations = { 
    "sigmoid":    (sigmoid, sigmoid_der),
    "ReLU":       (ReLU, ReLU_der),
    "leaky ReLU": (leaky_ReLU, leaky_ReLU_der)}

# nodes in each hidden layer to be tested
architectures = [[10], [50], [100], [50, 50], [100, 100], [50] * 3, [50] * 5, [50] * 10]
arch_labels = ['10', '50', '100', '50-50', '100-100', '3x50', '5x50', '10x50']

## Train all combinations

For every activation function and architecture we train a network and store the training and test MSE.
All networks are trained with Adam ($\gamma = 0.01$), 500 epochs and batch size 14.

In [ ]:
# Train and test MSE for each activation function and architecture combination
train_mse = np.zeros((len(activations), len(architectures)))
test_mse  = np.zeros((len(activations), len(architectures)))
# Learning curves and predictions for each activation function and architecture combination
results = {}

# Loop over activation functions
for i, (act_name, (act, act_der)) in enumerate(activations.items()):
    # Loop over architectures
    for j, hidden in enumerate(architectures):
        nodes = len(hidden) # Number of hidden layers
        # Create a new neural network for each combination of activation function and architecture
        net = NeuralNetwork(
            network_input_size=1,
            layer_output_sizes=hidden + [1],
            activation_funcs=[act] * nodes + [identity],
            activation_ders=[act_der] * nodes + [identity_der],
            cost_fun=mse,
            cost_der=mse_der,
            optimizer=Adam(gamma=0.01),     # new optimizer for every network
            seed=2026)
        # Train on the scaled training data
        history = net.train(X_train_s, y_train, epochs=500, batch_size=14,
                            rng=np.random.default_rng(2026))
        # MSE on training and test data
        train_mse[i, j] = mse(net.predict(X_train_s), y_train)
        test_mse[i, j]  = mse(net.predict(X_test_s), y_test)
        # Store learning curve and predicted curve for plotting 
        results[(act_name, arch_labels[j])] = {
            "history": history,
            "plot": net.predict((x_plot - x_mean) / x_std)}
    

## Test and training MSE

Test and training MSE for every combination of activation function and architecture. If the training MSE is much lower than the test MSE, the network is overfitting. The data have noise with $\sigma = 0.1$, so the lowest MSE we can expect is the noise floor $\sigma^2 = 0.01$.

In [ ]:
# Creating heatmap for the test MSE
fig, ax = heatmap(test_mse, range(len(architectures)), range(len(activations)),
                  'hidden layers (nodes per layer)', 'activation', label="Test MSE")
ax.set_xticklabels(arch_labels, rotation=45)            # architecture labels on x-axis
ax.set_yticklabels(list(activations), rotation=0)       # activation names on the y-axis
fig.set_size_inches(8, 4)
save_fig("c_heatmap_test")

In [ ]:
# Creating heatmap for the train MSE
fig, ax = heatmap(train_mse, range(len(architectures)), range(len(activations)),
                  'hidden layers (nodes per layer)', 'activation', label="Train MSE")
ax.set_xticklabels(arch_labels, rotation=45)
ax.set_yticklabels(list(activations), rotation=0)
fig.set_size_inches(8, 4)
save_fig("c_heatmap_train")

## The deepest networks: fits and learning curves

Fitted curves and learning curves for the deepest networks, with 10 hidden layers with 50 nodes each.

In [ ]:
# Predicted curves and learning curves for the deepest networks (10 hidden layers with 50 nodes each)
fig, ax = line_plot(
    x_plot,
    {
        "True Runge function": y_true,
        "NN 10x50, sigmoid":    results[("sigmoid", "10x50")]["plot"],
        "NN 10x50, ReLU":       results[("ReLU", "10x50")]["plot"],
        "NN 10x50, leaky ReLU": results[("leaky ReLU", "10x50")]["plot"]
    },
    xlabel="$x$", 
    ylabel="$y$", 
    data=(X_train, y_train), 
    name="c_fit_10x50")

epochs = np.arange(1, 501)
fig, ax = line_plot(
    epochs,
    {
        "sigmoid":    results[("sigmoid", "10x50")]["history"],
        "ReLU":       results[("ReLU", "10x50")]["history"],
        "leaky ReLU": results[("leaky ReLU", "10x50")]["history"]
    },
    xlabel="Epoch", 
    ylabel="Training MSE", 
    logy=True, 
    hlines={"noise floor": 0.01},
    name="c_learning_curve_10x50")


## Vanishing gradients

The size of $\partial C/\partial W$ in each layer of an untrained network with 10 hidden layers of 50 nodes. Backpropagation multiplies by the derivative of the activation function once per layer. For the sigmoid, $\sigma'(z) \le 1/4$, so the gradient shrinks on its way back to the first layers.


In [ ]:
# Gradient size for each layer of an untrained 10x50 network, for each activation function
grad_norms = {}

for act_name, (act, act_der) in activations.items():
    # New untrainted network with 10 hidden layers of 50 nodes each
    net = NeuralNetwork(
        network_input_size=1,
        layer_output_sizes=[50] * 10 + [1],
        activation_funcs=[act] * 10 + [identity],
        activation_ders=[act_der] * 10 + [identity_der],
        cost_fun=mse,
        cost_der=mse_der,
        optimizer=Adam(gamma=0.01),
        seed=2026)
    # Gradients of the cost function with respect to the weights and biases for each layer (backpropagation)
    grads = net.compute_gradient(X_train_s, y_train)
    # Size (norm) of the weight gradient dW in each layer
    grad_norms[act_name] = [np.linalg.norm(dW) for dW, db in grads]

# Print the gradient size layer by layer
    print(act_name)
    for l, g in enumerate(grad_norms[act_name], start=1):
        print(f"  layer {l:2d}:  |dC/dW| = {g:.1e}")
# Plot the gradient size against the layer number (log scale on y-axis), the ReLU and leaky ReLU curves overlap
fig, ax = line_plot(np.arange(1, 12), grad_norms, xlabel="layer", ylabel=r"$|\partial C/\partial W|$",
                    logy=True, name="c_gradient_norms")
